# Initialize dask cluster for **L0**

In [1]:
import dask, platform
print(f"Python version: {platform.python_version()}")
print(f"Dask version: {dask.__version__}")

# This notebook should be called from the right Jupyter virtual environment kernel.
# You can change this notebook kernel using the list on the top-right corner of this page.
assert platform.python_version() == "3.13.12"
assert dask.__version__ == "2026.3.0"

Python version: 3.13.12
Dask version: 2026.3.0


In [2]:
# Parameters. NOTE: the cell must be tagged as "parameters" !
# See: https://papermill.readthedocs.io/en/latest/usage-parameterize.html
scale=1
image_name="ghcr.io/rs-python/dask/l0/k8s"
image_tag="latest"
cluster_label="" # by default: <proc>.<user>.<docker-tag>
worker_cores=4
worker_memory=58
scheduler_memory_limit=60
big_resources=True

In [3]:
# Init cluster from this venv kernel
from resources.dask_clusters.dask_utils import *
from resources.dask_clusters.dask_venv import *
from resources.dask_clusters.cluster_config import * # see cluster_config.__init__

gateway, cluster, client = init_dask_cluster_l0_venv(
    scale=scale,
    image=f"{image_name}:{image_tag}",
    cluster_label=cluster_label,
    worker_cores=worker_cores,
    worker_memory=worker_memory,
    scheduler_memory_limit=scheduler_memory_limit,    
    worker_extra_pod_config=dpr_scheduler_pod_config if big_resources else dpr_worker_pod_config,
    scheduler_extra_pod_config=dpr_scheduler_pod_config,
    worker_extra_container_config=dpr_container_config,
    scheduler_extra_container_config=dpr_container_config,
)
display(cluster)

[dpr_container_config] Resolved configuration: {'volumeMounts': [{'name': 'rspython-dev-cs-01', 'mountPath': '/mnt/share/cs-01/', 'readOnly': False}]}
{
  "scale": 1,
  "image": "ghcr.io/rs-python/dask/l0/k8s:latest",
  "python_packages": [
    "eopf",
    "l0"
  ],
  "cluster_label": "dask-l0.jgaucher.latest",
  "worker_cores": 4,
  "worker_memory": 58,
  "scheduler_memory_limit": 60,
  "worker_extra_pod_config": {
    "affinity": {
      "nodeAffinity": {
        "requiredDuringSchedulingIgnoredDuringExecution": {
          "nodeSelectorTerms": [
            {
              "matchExpressions": [
                {
                  "key": "node-role.kubernetes.io/dask_scheduler",
                  "operator": "Exists"
                }
              ]
            }
          ]
        }
      }
    },
    "tolerations": [
      {
        "key": "role",
        "operator": "Equal",
        "value": "dask_scheduler",
        "effect": "NoSchedule"
      }
    ],
    "volumes": [
      {

/opt/conda/envs/py3.13.12-2026.3.0/lib/python3.13/site-packages/distributed/client.py:1607: VersionMismatchWarning: Mismatched versions found

+-------------+----------+-----------+---------+
| Package     | Client   | Scheduler | Workers |
+-------------+----------+-----------+---------+
| dask        | 2026.3.0 | 2026.7.1  | None    |
| distributed | 2026.3.0 | 2026.7.1  | None    |
| toolz       | 1.1.0    | 1.2.0     | None    |
+-------------+----------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-l0.jgaucher.latest' are up: 1/1
{
  "jupyter_token": "***",
  "dask_gateway_address": "http://dask-l0:8000",
  "cluster_label": "dask-l0.jgaucher.latest",
  "cluster_instance": "0fc835dc5c66497d8f6fd51fdbaa9678"
}
Python package versions (pip freeze):
[
  "eopf==3.0.0rc4",
  "l0 @ git+https://gitlab.eopf.copernicus.eu/L0/l0@48050c5a09cd4a8c95c5d56105166440f9e86325"
]


## Shutdown dask cluster

In [4]:
# Choose to shutdown the dask cluster
shutdown_checkbox

Checkbox(value=False, description='Shutdown the dask clusters', indent=False)

In [5]:
if shutdown_checkbox.value:
    shutdown_dask_clusters(gateway, cluster.name)
    close_dask_clusters(gateway, cluster, client)

## Only from command line

In [ ]:
print(KEEP_THIS_NOTEBOOK_OPEN)
!tail -f /dev/null

Keep this notebook open (when called from command line)
